[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C11_RAG_Retrieval_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy / pandas、CPU 可跑**，把一条 RAG 流水线（嵌入→检索→重排→生成→评测）从零搭出来，并与朴素参考 **对拍**。

这个 notebook 做三件事：① 确认环境；② 用一个最小例子体会「**语义检索 vs 词法检索**」的差别；③ 立下全课的纪律——**对拍（differential testing）**。

## 1 · 环境自检

只需要 `numpy` 与 `pandas`。`matplotlib` 可选（仅用于画召回-延迟 / 位置偏置曲线）。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
import pandas as pd
print('numpy', np.__version__, '| pandas', pd.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 一眼看懂语义检索 vs 词法检索

**词法检索**（如关键词重叠）匹配字面；**语义检索**（embedding 近邻）匹配意思。

下面用一个最小例子：查询「汽车」，候选里有「automobile（同义、零词重叠）」和「香蕉（无关）」。我们各用一个**玩具**词法分（词重叠）和一个**玩具**语义分（手工编的向量）看二者差别。

In [ ]:
# 玩具语料：每条文档手工给一个 2 维“语义向量”（仅为演示，真实是模型编码）
docs = ['一辆红色的汽车', 'a fast automobile on the road', '一根黄色的香蕉']
vecs = np.array([
    [1.0, 0.1],   # 汽车  -> 偏“交通工具”维
    [0.9, 0.2],   # automobile -> 几乎同方向（语义近！）
    [0.1, 1.0],   # 香蕉  -> 偏“水果”维
])
q_text, q_vec = '汽车', np.array([1.0, 0.15])

def lexical_overlap(q, d):
    # 玩具词法分：字符级重叠（中文按字，英文整体算 0）—— 故意让同义但零字面重叠的得 0
    return len(set(q) & set(d))

def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

lex = [lexical_overlap(q_text, d) for d in docs]
sem = [cosine(q_vec, v) for v in vecs]
df = pd.DataFrame({'doc': docs, '词法分(字重叠)': lex, '语义分(余弦)': np.round(sem, 3)})
print(df.to_string(index=False))
# 语义检索把 automobile 排在香蕉之上；词法检索看不出 automobile 与“汽车”相关
assert sem[1] > sem[2], '语义上 automobile 应比香蕉更接近“汽车”'
assert lexical_overlap(q_text, docs[1]) == 0, '同义但零字面重叠 -> 词法分=0'
print('\n✅ 语义检索抓住了“汽车≈automobile”，而词法检索对同义改写无能为力 —— 这就是模块 01 的主题')

## 3 · 这条流水线长什么样

把全课要搭的东西先跑一个**极简全链路**：3 条文档、1 个查询，做①向量检索 ②取 top-2 当上下文 ③一个玩具“生成” ④评一个 faithfulness。

现在不求精，只为看清 **嵌入→检索→生成→评测** 的骨架。后面六个模块逐环做深。

In [ ]:
# ① 嵌入（这里直接用上面的玩具向量）；② 检索 top-2
def normalize(M):
    return M / (np.linalg.norm(M, axis=-1, keepdims=True) + 1e-12)

D = normalize(vecs); q = normalize(q_vec[None])[0]
scores = D @ q                      # 归一化后点积 = 余弦
topk = np.argsort(-scores)[:2]
retrieved = [docs[i] for i in topk]
print('检索到的上下文 top-2:', retrieved)

# ③ 玩具“生成”：直接复述最相关文档（真实是 LLM 据上下文作答）
answer = retrieved[0]

# ④ 玩具 faithfulness：答案的字是否都被检索上下文覆盖（真实用 NLI/LLM 判蕴含）
ctx_chars = set(''.join(retrieved))
faithful = all(ch in ctx_chars for ch in answer)
print('答案:', answer, '| faithful(被上下文支持)?', faithful)
assert topk[0] == 0, '查询“汽车”最该检索到“汽车”那条'
assert faithful, '复述检索内容的答案必然忠实于上下文'
print('\n✅ 跑通了 嵌入→检索→生成→评测 的骨架；六个模块就是把每一环做深')

## 4 · 立纪律：对拍（differential testing）

本课每个**近似/优化**组件都要和一个**朴素参考**比对。比如近似最近邻(ANN)的质量，就用它与**暴力精确 kNN** 的 top-k **重合度**（召回率）来量化。

先把这个工作流跑通：写一个暴力 kNN 当参考，再写一个“只看一半库”的劣化版，量化它丢了多少召回。

In [ ]:
rng = np.random.default_rng(0)
X = rng.standard_normal((200, 16))      # 200 条 16 维库向量
qv = rng.standard_normal(16)

def knn_bruteforce(X, q, k=10):
    d = np.linalg.norm(X - q, axis=1)    # 精确欧氏距离
    return np.argsort(d)[:k]

def knn_half(X, q, k=10):
    # 劣化“近似”：只在前一半库里找（模拟只搜部分候选）
    half = X.shape[0] // 2
    d = np.linalg.norm(X[:half] - q, axis=1)
    return np.argsort(d)[:k]

def recall_at_k(approx_idx, exact_idx):
    return len(set(approx_idx) & set(exact_idx)) / len(exact_idx)

exact = knn_bruteforce(X, qv, k=10)
approx = knn_half(X, qv, k=10)
r = recall_at_k(approx, exact)
print(f'暴力 top-10 索引: {exact.tolist()}')
print(f'只看一半 top-10 : {approx.tolist()}')
print(f'召回率 recall@10 = {r:.2f}')
assert 0.0 <= r <= 1.0
assert set(approx).issubset(set(range(100))), '只看前一半，结果必来自前 100 条'
print('\n✅ 这就是全课评 ANN 的工作流：近似结果 vs 暴力参考 -> 召回率。结构正确则数字可信。')

## 5 · 一个会贯穿全课的对拍工具

把「召回率对拍」封装成小函数，后面模块 02 评 IVF/HNSW/PQ 都用它。它是本课所有 `assert` 背后的统一裁判之一。

In [ ]:
def check_recall(name, approx_idx, exact_idx, min_recall=0.0):
    '''对拍：近似检索 vs 暴力参考。打印并断言召回率达标。'''
    r = len(set(approx_idx) & set(exact_idx)) / max(len(exact_idx), 1)
    print(f'[{name:<24}] recall@{len(exact_idx)} = {r:.3f}')
    assert r >= min_recall, f'{name} 召回率 {r:.3f} 低于要求 {min_recall}'
    return r

# 演示：暴力 vs 自己（应当 recall=1.0）
check_recall('bruteforce vs self', exact, exact, min_recall=1.0)
check_recall('half-search', approx, exact, min_recall=0.0)
print('\n这就是全课的工作流：搭组件 -> 对拍朴素参考 -> assert 兜底。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你在 numpy/pandas 里搭的每个检索/重排/评测组件，都会用对拍或按定义直算来验证；结构正确则数字可信，数字可信则结论可迁移到真实系统。

**接下来六个模块**：01 嵌入与语义搜索 → 02 向量检索 → 03 重排 → 04 RAG 评测 → 05 检索指标 → 06 长上下文评测。每一步都建立在前一步之上。

下一站：**模块 01 · 嵌入与语义搜索**。